In [15]:
import pandas as pd
import numpy as np
import re
import os
from sklearn.model_selection import train_test_split

In [16]:
df = pd.read_csv("../data/raw/PRDECT-ID_Dataset.csv")
df.head()

,Category,Product Name,Location,Price,Overall Rating,Number Sold,Total Review,Customer Rating,Customer Review,Sentiment,Emotion
0,Computers and Laptops,Wireless Keyboard i8 Mini TouchPad Mouse 2.4G ...,Jakarta Utara,53500,4.9,5449,2369,5,Alhamdulillah berfungsi dengan baik. Packaging...,Positive,Happy
1,Computers and Laptops,PAKET LISENSI WINDOWS 10 PRO DAN OFFICE 2019 O...,Kota Tangerang Selatan,72000,4.9,2359,1044,5,"barang bagus dan respon cepat, harga bersaing ...",Positive,Happy
2,Computers and Laptops,SSD Midasforce 128 Gb - Tanpa Caddy,Jakarta Barat,213000,5.0,12300,3573,5,"barang bagus, berfungsi dengan baik, seler ram...",Positive,Happy
3,Computers and Laptops,ADAPTOR CHARGER MONITOR LCD LED TV LG merek LG...,Jakarta Timur,55000,4.7,2030,672,5,bagus sesuai harapan penjual nya juga ramah. t...,Positive,Happy
4,Computers and Laptops,ADAPTOR CHARGER MONITOR LCD LED TV LG merek LG...,Jakarta Timur,55000,4.7,2030,672,5,"Barang Bagus, pengemasan Aman, dapat Berfungsi...",Positive,Happy


In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5400 entries, 0 to 5399
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Category         5400 non-null   object 
 1   Product Name     5400 non-null   object 
 2   Location         5400 non-null   object 
 3   Price            5400 non-null   int64  
 4   Overall Rating   5400 non-null   float64
 5   Number Sold      5400 non-null   int64  
 6   Total Review     5400 non-null   int64  
 7   Customer Rating  5400 non-null   int64  
 8   Customer Review  5400 non-null   object 
 9   Sentiment        5400 non-null   object 
 10  Emotion          5400 non-null   object 
dtypes: float64(1), int64(4), object(6)
memory usage: 464.2+ KB


In [18]:
df = df[["Customer Review", "Emotion"]]
df = df.rename(columns={'Customer Review': 'text', 'Emotion': 'label'})

df.shape

(5400, 2)

In [19]:
print("Before dedup:", df.shape)
df = df.drop_duplicates()                          
df = df.drop_duplicates(subset=["text"])           
print("After dedup:", df.shape)

Before dedup: (5400, 2)
After dedup: (5305, 2)


In [20]:
def clean_text(text):
    text = str(text)
    
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'([!?.]){2,}', r'\1', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s!?.]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = text.lower()
    
    return text

df["text"] = df["text"].apply(clean_text)

In [21]:
df = df[df["text"].str.strip() != ""]
df = df[df["text"].str.split().str.len() >= 2]
print("Shape after cleaning:", df.shape)

Shape after cleaning: (5253, 2)


In [22]:
unique_labels = sorted(df["label"].unique())
label_mapping = {label: idx for idx, label in enumerate(unique_labels)}
id_to_label = {idx: label for label, idx in label_mapping.items()}

print("Label mapping:", label_mapping)
df["label"] = df["label"].map(label_mapping)

Label mapping: {'Anger': 0, 'Fear': 1, 'Happy': 2, 'Love': 3, 'Sadness': 4}


In [23]:
train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["label"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df["label"]
)

print("Train: ", train_df.shape)
print("Val: ", val_df.shape)
print("Test: ", test_df.shape)
print("\nTrain label distribution:")
print(train_df["label"].value_counts().sort_index())

Train:  (4202, 2)
Val:  (525, 2)
Test:  (526, 2)

Train label distribution:
label
0     534
1     713
2    1395
3     619
4     941
Name: count, dtype: int64


In [24]:
save_dir = "../data/processed"

os.makedirs(save_dir, exist_ok=True)

train_df.to_csv(os.path.join(save_dir, "train.csv"), index=False)
val_df.to_csv(os.path.join(save_dir, "val.csv"), index=False)
test_df.to_csv(os.path.join(save_dir, "test.csv"), index=False)

# Save label mapping for use in modeling notebooks
import json
with open(os.path.join(save_dir, "label_mapping.json"), "w") as f:
    json.dump(label_mapping, f)

In [25]:
print("Sample cleaned reviews:")
for _, row in train_df.sample(5, random_state=1).iterrows():
    print(f"[{id_to_label[row['label']]}] {row['text']}")

Sample cleaned reviews:
[Fear] gagang tlp ny patah
[Fear] barang tidak sesuai pesanan . mohon pendataan barang lebih cermat lagi
[Happy] pengiriman cepat bgt.mantap. sukses selalu yaaa
[Happy] bagus simple cuma menurut saya kurang keras suaranya. tapi bagus ?
[Sadness] aku baru pakai sebentar patah


In [26]:
print(label_mapping)
print(id_to_label)

{'Anger': 0, 'Fear': 1, 'Happy': 2, 'Love': 3, 'Sadness': 4}
{0: 'Anger', 1: 'Fear', 2: 'Happy', 3: 'Love', 4: 'Sadness'}


In [27]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.array(sorted(df["label"].unique()))
weights = compute_class_weight(class_weight="balanced", classes=classes, y=train_df["label"])
class_weights = dict(zip(classes, weights))
print(class_weights)

clean_class_weights = {str(key): float(value) for key, value in class_weights.items()}

with open(os.path.join(save_dir, "class_weights.json"), "w") as f:
    json.dump(clean_class_weights, f, indent=4)

{0: 1.5737827715355805, 1: 1.1786816269284712, 2: 0.6024372759856631, 3: 1.3576736672051697, 4: 0.8930924548352817}
